In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

In [ ]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input/fields', TASK/'output'
FIELDS = {
    'biological_sciences': 'Biological sciences', 'chemistry': 'Chemistry',
    'computer_science': 'Computer science (AARC)', 'economics': 'Economics',
    'electrical_engineering': 'Electrical engineering', 'history': 'History',
    'management': 'Management', 'mathematics': 'Mathematics',
    'mechanical_engineering': 'Mechanical engineering', 'physics': 'Physics',
    'political_science': 'Political science', 'psychology': 'Psychology', 'sociology': 'Sociology'}
ALIASES = {'biological_sciences': ['biological-sciences-general'],
           'economics': ['economics-general'], 'physics': ['physics-general'],
           'psychology': ['psychology-general'],
           'sociology': ['sociology-articles-focused']}  # both labels use AARC ID 156
LONG_COLUMNS = ['PersonId', 'CareerAgeZero', 'pubs_adj']
WINDOW, MIN_END_AGE = 4, 4  # keep the Shockley notebook's career-age cutoff

In [ ]:
def read_trajectories(path):
    data = pd.read_csv(path, dtype={'dblp_id': 'string', 'PersonId': 'string'})
    if 'dblp_id' in data:
        ages = [column for column in data if column.isdecimal()]
        panel = data.set_index('dblp_id')[ages]
        panel.columns = panel.columns.astype(int)
    else:
        panel = data[LONG_COLUMNS].pivot(index='PersonId', columns='CareerAgeZero', values='pubs_adj')
    panel.index.name = 'researcher'
    return panel

files = {field: [INPUT/f'secret_{slug}_trajectories.csv' for slug in [field.replace('_', '-'), *ALIASES.get(field, [])]] for field in FIELDS}

In [ ]:
rows = []
for field, label in FIELDS.items():
    paths = [path for path in files[field] if path.is_file()]
    row = {'field': field, 'label': label, 'n_eligible': 0, 'n_positive': 0, 'qq_r': np.nan, 'status': 'missing_field_export', 'source_files': '; '.join(path.name for path in paths)}
    rows.append(row)
    if not paths:
        continue

    panel = pd.concat([read_trajectories(path) for path in paths])
    panel = panel.reset_index().drop_duplicates().set_index('researcher')
    if panel.index.has_duplicates:
        raise ValueError(f'{label}: conflicting trajectories for a researcher across field exports')

    panel = panel.sort_index(axis=1, ascending=False)
    observed = panel.notna()
    window = observed & observed.cumsum(axis=1).le(WINDOW)
    eligible = window.sum(axis=1).eq(WINDOW) & observed.loc[:, panel.columns >= MIN_END_AGE].any(axis=1)
    totals = panel.where(window).sum(axis=1, min_count=WINDOW).loc[eligible]
    positive = totals.loc[totals.gt(0) & np.isfinite(totals)]
    row.update(n_eligible=len(totals), n_positive=len(positive), status='insufficient_positive_totals')
    if positive.nunique() >= 2:
        _, (_, _, r) = stats.probplot(np.log2(positive), dist='norm')
        row.update(qq_r=r, status='ok')

In [ ]:
summary = pd.DataFrame(rows).sort_values(['qq_r', 'field']).reset_index(drop=True)
summary.to_csv(OUTPUT/'aarc_field_lognormality.csv', index=False)
display(summary)